# Athénée Investment — Third-Party Marketing & Capital Introduction Pipeline

This notebook provides an automated, institutional-grade pipeline to:
1. **Select one or multiple hedge funds** from `../data/hf_returns/` using their CSV names.
2. **Compute the complete `single_fund_analysis` quantitative indicator suite** (CAGR, Annualized Return, Annualized Volatility, Sharpe Ratio, Sortino Ratio, Maximum Drawdown, Calmar Ratio, Positive Months Hit Rate, Crisis Hit Rate, Upside/Downside Capture, OLS Annualized Alpha & $t$-stat, Beta & $t$-stat, Pearson & Spearman Correlations with 95% Standard & Moving-Block Bootstrap Confidence Intervals, Tail Correlations in S&P 500 Q1/D1 Drawdowns, Cross-Asset WTI & BTC Betas, and Ljung-Box Serial Correlation Diagnostics).
3. **Build the Monthly & Yearly Return Matrix** (Years as rows, Jan–Dec as columns, and compounded Yearly performance as the final column).
4. **Compile a modern 2-page LaTeX PDF Tear Sheet** per fund (`output/<fund>_tearsheet.pdf`):
   - **Page 1 (One-Pager Tear Sheet):** Athénée Investment Logo, Hero KPI Cards, Full Quantitative Indicators Table (Fund vs. S&P 500), and Color-Coded Monthly Return Matrix.
   - **Page 2 (Appendix & Risk / Legal Disclosures):** Extended Bootstrap & Factor Diagnostics, Quantitative Methodology Definitions, and Capital Introduction / Third-Party Marketing Legal Disclaimer & Risk Warnings.

In [ ]:
import os
import pandas as pd
from IPython.display import display, HTML

from pipeline import (
    list_available_funds,
    run_fund_pipeline,
    run_batch_pipeline,
    load_fund_and_benchmarks,
    compute_single_fund_metrics,
    build_monthly_return_matrix,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

available_funds = list_available_funds()
print(f"Total funds available in ../data/hf_returns/: {len(available_funds)}")
print(available_funds)

## 1. Select Funds by CSV Name
Specify either:
- A list of one or more CSV names (with or without `.csv`), e.g. `['dcm_systematic', 'smithson', '26_miles']`
- Or `'all'` to generate PDF tear sheets for every fund in `../data/hf_returns/` in one shot.

In [ ]:
# ===========================================================================
# SELECT YOUR FUNDS HERE (USING THEIR CSV NAME FROM data/hf_returns/)
# ===========================================================================
SELECTED_FUNDS = [
    'dcm_systematic',
    'smithson',
    '26_miles',
    'adar1',
    'h3im',
    'sam_capital',
]

# Optional date range filter (set to None to use each fund's full available history)
START_DATE = None  # e.g. '2022-01-01'
END_DATE = None    # e.g. '2026-07-01'

# Optional per-fund overrides (custom display title, strategy subtitle, or leverage)
FUND_CONFIGS = {
    # 'dcm_systematic': {
    #     'title': 'DCM Systematic',
    #     'subtitle': 'Systematic Multi-Asset & Diversified Quantitative Strategy',
    #     'leverage': 1.0,
    # }
}

OUTPUT_DIR = './output'

## 2. Run the Batch Pipeline (Compute Metrics, Return Matrix & Compile LaTeX PDFs)
Running the cell below processes all selected funds using the exact same model and saves:
- `./output/<fund>_tearsheet.pdf` (2-Page PDF: One-Pager Tear Sheet + Appendix & Risk Disclaimer)
- `./output/<fund>_tearsheet.tex` (Modern LaTeX source)
- `./output/<fund>_return_matrix.csv` (Year $\times$ Month + Yearly performance table)
- `./output/<fund>_metrics.csv` (Full `single_fund_analysis` metrics)
- `./output/summary_selected_funds_metrics.csv` (Combined comparison table across all selected funds)

In [ ]:
metrics_list = run_batch_pipeline(
    selected_funds=SELECTED_FUNDS,
    output_dir=OUTPUT_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
    fund_configs=FUND_CONFIGS,
    compile_pdf=True,
)

summary_df = pd.DataFrame(metrics_list)
if not summary_df.empty:
    display_cols = [
        'fund_col', 'start_date', 'end_date', 'n_months',
        'cagr', 'ann_vol', 'sharpe', 'sortino', 'mdd', 'calmar',
        'fund_pos_rate', 'crisis_hit_rate', 'up_capture', 'down_capture',
        'alpha', 'alpha_tstat', 'beta', 'pearson_corr', 'spearman_corr'
    ]
    display(summary_df[display_cols])

## 3. Inspect a Single Fund's Return Matrix & Detailed Indicators

In [ ]:
INSPECT_FUND = SELECTED_FUNDS[0]
single_res = run_fund_pipeline(INSPECT_FUND, output_dir=OUTPUT_DIR, compile_pdf=False)

matrix_df = pd.read_csv(single_res['matrix_csv_path'], index_col=0)
display(HTML(f"<h3>Monthly & Yearly Return Matrix — {INSPECT_FUND}</h3>"))
display(matrix_df.map(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "-"))